## Azure OpenAI API test

In [ ]:
# Azure OpenAI API test

import os
from openai import AzureOpenAI

os.environ["OPENAI_AZURE_API_KEY"]
endpoint = "https://rt-bdi-zanolo.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o"

subscription_key = os.environ["OPENAI_AZURE_API_KEY"]
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "I am going to Paris, what should I see?",
        }
    ],
    max_tokens=4096,
    temperature=1.0,
    top_p=1.0,
    model=deployment
)

print(response.choices[0].message.content)

# Test Gymnasium


In [ ]:
import gymnasium as gym
import imageio
import os
import gymnasium_robotics

env = gym.make("CartPole-v1", render_mode="rgb_array")
# The CartPole environment: balance a pole on a moving cart
env

In [ ]:

GAME_NAME = "Taxi-v3"
VIDEO_PATH = F"../data/agent_videos/{GAME_NAME}_test.mp4"
gym.register_envs(gymnasium_robotics)
env = gym.make(GAME_NAME, render_mode="rgb_array")
obs, info = env.reset()

frames = []
done = False
i = 0
while not done:
    i += 1
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    frames.append(env.render())  # frame RGB (numpy array)
    done = terminated or truncated
    if i == 10:
        done = True
env.close()
imageio.mimsave(VIDEO_PATH, frames, fps=30, macro_block_size=1)
print(F"Video saved at: {VIDEO_PATH}")


In [ ]:
gym.pprint_registry()

In [ ]:
# --------------------------------------------------------------------------
# Example usage – the agent (or main loop) controls the interaction
# --------------------------------------------------------------------------
from environments import GymEnvironment, GymEnvConfig
from datetime import datetime

class RandomAgent:
    def __init__(self, action_space):
        self.action_space = action_space
        self.observations_count = 0
        
    def act(self, observation, info = None):
        self.observations_count += 1
        print(f"{self.observations_count}: Observation: {observation} - {info}")
        return self.action_space.sample()
    
env_id="FrozenLake-v1"
cfg = GymEnvConfig(env_id=env_id, kwargs={"is_slippery": False}, render_mode="rgb_array")
env = GymEnvironment(cfg)
agent = RandomAgent(env.action_space)

stats = env.run_episode(agent, record_path=f"../data/agent_videos/{env_id}_{datetime.now()}.mp4")
print(stats)

env.close()

# Test OSWorld

In [ ]:
from desktop_env.desktop_env import DesktopEnv
from pathlib import Path
import yaml
from myagent.utility import print_dict
from myagent.pydantic_models import OSWorldConfig

SETTINGS_PATH = Path("../script/config_files/osworld_settings.yaml")

def load_settings() -> dict:
    if not SETTINGS_PATH.exists():
        raise FileNotFoundError("Missing osworld_settings.yaml")
    with SETTINGS_PATH.open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    return OSWorldConfig(**cfg)  

cfg = load_settings()
print_dict(cfg.model_dump(), title="Bridge Settings:")

env = DesktopEnv(provider_name=cfg.provider_name, 
                        path_to_vm=cfg.vmx, 
                        headless=cfg.headless, 
                        require_terminal=cfg.require_terminal,
                        require_a11y_tree=cfg.require_a11y_tree,
                        action_space=cfg.action_space)

ModuleNotFoundError: No module named 'desktop_env'

In [ ]:
o = env._observe()
o

In [ ]:
# --- TEST 1: /info + screenshot fallback ---
info = requests.get(f"{BASE_URL}/info", timeout=5).json()
print("INFO:", info)

frame = env.render_frame()  # usa /screenshot del bridge
print("Frame shape:", frame.shape)
show_ndarray(frame)


In [ ]:
# --- TEST 2: observe con a11y e rendering nel notebook ---
# Abilita a11y nella config a runtime
env.settings.want_a11y = True

# Usa reset() che internamente chiama /task/reset (soft) e poi /observe
obs = env.reset()

# Mostra lo screenshot se presente
if "image" in obs:
    print("Screenshot acquisito.")
    show_ndarray(obs["image"])

# Estrai a11y
a11y = obs.get("a11y")
if a11y is None:
    print("A11y non disponibile (server guest non la espone o disattivata).")
else:
    # a11y può essere JSON (dict) o XML (string)
    if isinstance(a11y, dict):
        print("A11y in formato JSON:")
        display(JSON(a11y))
        # stampa qualche dettaglio
        print("Chiavi top-level:", list(a11y.keys())[:10])
    elif isinstance(a11y, str):
        print("A11y in formato XML, prime ~1000 battute:")
        print(textwrap.shorten(a11y, width=1000, placeholder=" ..."))
    else:
        print("Formato a11y sconosciuto:", type(a11y))


In [ ]:
# --- TEST 3: normalizzazione a11y (solo se è JSON) ---
from collections import deque
import pandas as pd

def flatten_a11y_json(a11y_json):
    """
    Tenta di flattenare un albero a11y generico in righe (role, name, bounds, children_count).
    Adatta i campi se i nomi nel tuo server guest differiscono.
    """
    rows = []
    q = deque()
    q.append(a11y_json)
    while q:
        node = q.popleft()
        role = node.get("role") or node.get("type") or node.get("class") or "unknown"
        name = node.get("name") or node.get("label") or node.get("text") or ""
        # bounds/bounding box potrebbero essere in diversi formati
        bounds = node.get("bounds") or node.get("bbox") or node.get("rect") or node.get("geometry")
        # children possono essere "children", "nodes", "children_list", ecc.
        children = node.get("children") or node.get("nodes") or []
        rows.append({
            "role": role,
            "name": name,
            "bounds": bounds,
            "children_count": len(children)
        })
        for c in children:
            q.append(c)
    return pd.DataFrame(rows)

a11y = obs.get("a11y")
if isinstance(a11y, dict):
    df = flatten_a11y_json(a11y)
    print("Totale nodi:", len(df))
    display(df.head(20))
else:
    print("A11y non in JSON (o non disponibile), salto la tabellina.")


In [ ]:
# --- TEST 4: azioni base tramite step() ---
env.settings.want_a11y = False  # per ridurre overhead durante azioni

# 4a) muovi e clicca
obs, done, info = env.step([
    {"type": "move", "x": 0, "y": 0},
    {"type": "click", "button": "left", "count": 1},
])
print("Click info:", info)
show_ndarray(obs["image"])

# 4b) digita testo nel campo attivo
obs, done, info = env.step([
    {"type": "type", "text": "hello from agent"},
])
print("Type info:", info)
show_ndarray(obs["image"])

# 4c) hotkey (esempio generico: ENTER)
obs, done, info = env.step([
    {"type": "key", "combo": ["ENTER"]},
])
print("Key info:", info)
show_ndarray(obs["image"])

# 4d) scroll (su/giu)
obs, done, info = env.step([
    {"type": "scroll", "dx": 0, "dy": -600},
    {"type": "wait", "seconds": 0.2},
    {"type": "scroll", "dx": 0, "dy": 600},
])
print("Scroll info:", info)
show_ndarray(obs["image"])


In [ ]:
# --- TEST 5: drag & drop semplice ---
# Trascina una regione (coordinate d'esempio: adattale alla tua risoluzione)
obs, reward, done, info = env.step([
    {"type": "drag", "x0": 600, "y0": 400, "x1": 1100, "y1": 700, "button": "left"},
    {"type": "wait", "seconds": 0.1}
])
print("Drag info:", info)
show_ndarray(obs["image"])


In [ ]:
# --- TEST 6: piccolo loop con timing ---
import statistics as stats

latencies = []
for i in range(5):
    t0 = time.time()
    obs, *_ = env.step([
        {"type": "move", "x": 200 + i*50, "y": 200 + i*30},
        {"type": "click", "button": "left", "count": 1},
        {"type": "wait", "seconds": 0.05},
    ])
    latencies.append((time.time()-t0)*1000)
    print(f"Step {i}: frame {obs['image'].shape if 'image' in obs else None}")

print("Latency ms (min/avg/max):", round(min(latencies),1), "/", round(stats.mean(latencies),1), "/", round(max(latencies),1))


In [ ]:
# --- TEST 7: gestione errori azione sconosciuta ---
try:
    obs, reward, done, info = env.step([
        {"type": "unknown_action", "foo": 123}
    ])
    print("Risposta step:", info)
    if "image" in obs:
        show_ndarray(obs["image"])
except requests.HTTPError as e:
    # Se il bridge risponde 4xx/5xx
    print("HTTP error:", e.response.status_code, e.response.text)
except Exception as e:
    print("Errore generico:", e)


In [ ]:
# --- TEST 8: soft task reset ---
# Usa direttamente l'endpoint /task/reset per simulare un nuovo episodio
r = requests.post(f"{BASE_URL}/task/reset", json={"task_id": "smoke"}, timeout=5)
print("task_reset:", r.status_code, r.json())

# Poi prendi una nuova osservazione
env.settings.want_a11y = True
obs = env.reset()
print("Nuova oss. con a11y:", "a11y" in obs, "| image:", "image" in obs)
if "image" in obs: show_ndarray(obs["image"])

In [ ]:
# --- TEST 9: osservazione con sola a11y ---
env.settings.want_image = False
env.settings.want_a11y = True

obs = env.reset()
a11y = obs.get("a11y")
print("Solo a11y disponibile? ->", a11y is not None)

# Ripristina impostazioni standard
env.settings.want_image = True
env.settings.want_a11y = False


In [ ]:
# --- TEST 10: chiusura ---
env.close()
print("Env chiuso.")
